# Thư viện và dữ liệu

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [1]:
df_full = pd.read_csv('../data_raw/fact_sales_full.csv', parse_dates=['order_date'])
df_full.head()

# Đặc trưng RFM

In [1]:
rfm = df_full.groupby('customer_code').agg(
    recency   = ('order_date', lambda x: (df_full.order_date.max() - x.max()).days),
    frequency = ('so_number', 'nunique'),
    monetary  = ('line_total', 'sum')
).reset_index()
rfm['churn'] = (rfm.recency > 90).astype(int)
rfm.head()

# Mô hình XGBoost

In [1]:
params_clf = pd.read_csv('best_params_classifier.csv', index_col=0)
print(params_clf)

In [1]:
X = rfm[['recency','frequency','monetary']].values
y = rfm['churn'].values
clf = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                    eval_metric='logloss', random_state=42, verbosity=0)
cv_scores = cross_val_score(clf, X, y, cv=StratifiedKFold(3), scoring='roc_auc')
print(f'AUC-ROC: {cv_scores.mean():.3f}')
clf.fit(X, y)

# Xuất kết quả

In [1]:
rfm['churn_prob'] = clf.predict_proba(X)[:, 1]
churn_list = rfm.sort_values('churn_prob', ascending=False).head(20)
churn_list

In [1]:
pred_gbm = pd.read_csv('Ensemble/predictions_prophet.csv')
pred_gbm.to_csv('Ensemble/predictions_gbm.csv', index=False)
pred_gbm